In [1]:
import os
import torch
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import MultipleNegativesRankingLoss

MODEL_ID        = "nomic-ai/nomic-embed-text-v1.5"
DATASET_ID      = "code-search-net/code_search_net"
OUTPUT_DIR      = "./checkpoints/nomic-codesearch"
FINAL_MODEL_DIR = "./models/nomic-codesearch-finetuned"
TRAIN_SAMPLES   = 50_000
EVAL_SAMPLES    = 2_000


def build_pairs(split, raw_ds, max_samples):
    data = raw_ds[split].filter(
        lambda ex: len(ex["func_documentation_string"].strip()) >= 20
                   and len(ex["whole_func_string"].strip()) >= 50
    )
    data = data.select(range(min(max_samples, len(data))))
    return data.map(
        lambda ex: {
            "anchor":   ex["func_documentation_string"].strip(),
            "positive": ex["whole_func_string"].strip(),
        },
        remove_columns=data.column_names,
    )


def build_evaluator(eval_dataset):
    n = len(eval_dataset)
    return InformationRetrievalEvaluator(
        queries       = {str(i): eval_dataset[i]["anchor"]   for i in range(n)},
        corpus        = {str(i): eval_dataset[i]["positive"] for i in range(n)},
        relevant_docs = {str(i): {str(i)}                   for i in range(n)},
        name="codesearchnet-eval",
        batch_size=32,
        show_progress_bar=True,
    )


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def main():
    device  = get_device()
    is_cuda = device == "cuda"
    # if is_cuda:
    #     os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    print(f"Device: {device}")

    raw_ds        = load_dataset(DATASET_ID, "python")
    train_dataset = build_pairs("train",      raw_ds, TRAIN_SAMPLES)
    eval_dataset  = build_pairs("validation", raw_ds, EVAL_SAMPLES)
    print(f"Train: {len(train_dataset)}  |  Eval: {len(eval_dataset)}")

    # Load model with specific configurations for speed
    model = SentenceTransformer(
        MODEL_ID, 
        trust_remote_code=True,
        device=device
    )
    
    # Nomic 1.5 suggests a specific prompt for queries, but we'll stick to basic for now
    # or we can set max_seq_length if needed.
    model.max_seq_length = 512 # Code can be long, but 512 is a good balance for speed
    
    evaluator = build_evaluator(eval_dataset)

    print("Baseline NDCG@10:", evaluator(model))

    trainer = SentenceTransformerTrainer(
        model=model,
        args=SentenceTransformerTrainingArguments(
            output_dir                    = OUTPUT_DIR,
            num_train_epochs              = 2,
            per_device_train_batch_size   =  32, # Increased significantly for 20GB VRAM
            per_device_eval_batch_size    = 32,
            gradient_accumulation_steps   = 1, # Set to 1 since batch size is already large
            learning_rate                 = 2e-5,
            warmup_ratio                  = 0.1,
            lr_scheduler_type             = "cosine",
            fp16                          = is_cuda,
            bf16                          = False,
            eval_strategy                 = "steps",
            eval_steps                    = 100, # More frequent eval since steps will be faster
            save_strategy                 = "steps",
            save_steps                    = 100,
            save_total_limit              = 2,
            load_best_model_at_end        = True,
            metric_for_best_model         = "codesearchnet-eval_cosine_ndcg@10",
            logging_dir                   = "./logs",
            logging_steps                 = 10,
            report_to                     = "none",
            dataloader_num_workers        = 0, # Keep at 0 for Windows stability
            dataloader_pin_memory         = True if is_cuda else False,
            tf32                          = True if is_cuda else False, # Enable TF32 for Ampere+ GPUs
        ),
        train_dataset = train_dataset,
        eval_dataset  = eval_dataset,
        loss          = MultipleNegativesRankingLoss(model),
        evaluator     = evaluator,
    )

    trainer.train()
    model.save_pretrained(FINAL_MODEL_DIR)
    print(f"Done. Model saved to {FINAL_MODEL_DIR}")


if __name__ == "__main__":
    main()


c:\Users\1305m\anaconda3\envs\KeshavLLM\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Train: 50000  |  Eval: 2000


<All keys matched successfully>
Batches:   0%|          | 0/63 [00:00<?, ?it/s]C:\Users\1305m\.cache\huggingface\modules\transformers_modules\nomic-ai\nomic-bert-2048\7710840340a098cfb869c4f65e87cf2b1b70caca\modeling_hf_nomic_bert.py:1580: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = F.scaled_dot_product_attention(
Corpus Chunks: 100%|██████████| 1/1 [00:18<00:00, 18.21s/it]


Baseline NDCG@10: {'codesearchnet-eval_cosine_accuracy@1': 0.918, 'codesearchnet-eval_cosine_accuracy@3': 0.968, 'codesearchnet-eval_cosine_accuracy@5': 0.9735, 'codesearchnet-eval_cosine_accuracy@10': 0.9805, 'codesearchnet-eval_cosine_precision@1': 0.918, 'codesearchnet-eval_cosine_precision@3': 0.3226666666666666, 'codesearchnet-eval_cosine_precision@5': 0.1947, 'codesearchnet-eval_cosine_precision@10': 0.09805000000000001, 'codesearchnet-eval_cosine_recall@1': 0.918, 'codesearchnet-eval_cosine_recall@3': 0.968, 'codesearchnet-eval_cosine_recall@5': 0.9735, 'codesearchnet-eval_cosine_recall@10': 0.9805, 'codesearchnet-eval_cosine_ndcg@10': 0.9524539389403591, 'codesearchnet-eval_cosine_mrr@10': 0.9431134920634916, 'codesearchnet-eval_cosine_map@100': 0.943807805180771, 'codesearchnet-eval_dot_accuracy@1': 0.8725, 'codesearchnet-eval_dot_accuracy@3': 0.958, 'codesearchnet-eval_dot_accuracy@5': 0.971, 'codesearchnet-eval_dot_accuracy@10': 0.978, 'codesearchnet-eval_dot_precision@1': 0

c:\Users\1305m\anaconda3\envs\KeshavLLM\lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
  0%|          | 10/3126 [00:12<51:12,  1.01it/s] 

...

{'loss': 0.0, 'grad_norm': 0.00018862202705349773, 'learning_rate': 1.5964624012021478e-09, 'epoch': 1.99}


100%|█████████▉| 3120/3126 [1:31:15<00:07,  1.21s/it]

{'loss': 0.0, 'grad_norm': 0.0008589589269831777, 'learning_rate': 2.2450765885473347e-10, 'epoch': 2.0}


100%|██████████| 3126/3126 [1:31:27<00:00,  1.76s/it]


{'train_runtime': 5487.2544, 'train_samples_per_second': 18.224, 'train_steps_per_second': 0.57, 'train_loss': 0.0008254960795101638, 'epoch': 2.0}
Done. Model saved to ./models/nomic-codesearch-finetuned


In [14]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer("./models/nomic-codesearch-finetuned", trust_remote_code=True)

a = model.encode("def walk_files(dir): ...")
b = model.encode("def walk_dir(path): ...")

cos = torch.nn.functional.cosine_similarity(torch.tensor(a), torch.tensor(b), dim=0)
print(f"Cosine sim: {cos:.4f}")  # should be > 0.5 for a related pair

<All keys matched successfully>


Cosine sim: 0.7863


In [5]:
!pip install optimum[exporters] onnx onnxruntime



In [8]:
!optimum-cli export onnx \
  --model ./models/nomic-codesearch-finetuned \
  --task feature-extraction \
  --opset 17 \
  --trust-remote-code \
  ./models/nomic-codesearch-onnx/


<All keys matched successfully>
C:\Users\1305m\.cache\huggingface\modules\transformers_modules\nomic-ai\nomic-bert-2048\7710840340a098cfb869c4f65e87cf2b1b70caca\modeling_hf_nomic_bert.py:1386: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if seqlen > self._seq_len_cached:
C:\Users\1305m\.cache\huggingface\modules\transformers_modules\nomic-ai\nomic-bert-2048\7710840340a098cfb869c4f65e87cf2b1b70caca\modeling_hf_nomic_bert.py:1339: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  seqlen > self._seq_len_cached
C:\Users\1305m\.cache\huggingface\modules\transf

In [38]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="./models/nomic-codesearch-onnx/model.onnx",
    model_output="./models/nomic-codesearch-onnx/model_int8.onnx",
    weight_type=QuantType.QInt8,
    #optimize_model=True
)
print("Done.")

Done.


In [43]:
import onnxruntime as ort
import numpy as np
from datasets import load_dataset

# reuse your build_pairs and tokenizer setup
tokenizer = AutoTokenizer.from_pretrained("./models/nomic-codesearch-onnx")
eval_dataset = build_pairs("validation", load_dataset(DATASET_ID, "python"), 500)

sess_fp32 = ort.InferenceSession("./models/nomic-codesearch-onnx/model.onnx")
sess_int8 = ort.InferenceSession("./models/nomic-codesearch-onnx/model_int8.onnx")
def get_embeddings(sess, texts):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="np")
    
    # get only the input names the model actually expects
    valid_inputs = {inp.name for inp in sess.get_inputs()}
    feeds = {k: v.astype(np.int64) for k, v in enc.items() if k in valid_inputs}
    
    out = sess.run(None, feeds)[0]   # (batch, seq, 768)
    mask = enc["attention_mask"][:, :, None].astype(np.float32)
    return (out * mask).sum(1) / mask.sum(1) # mean pool

def cosine(a, b):
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))

anchors   = [eval_dataset[i]["anchor"]   for i in range(500)]
positives = [eval_dataset[i]["positive"] for i in range(500)]

cos_fp32 = cosine(get_embeddings(sess_fp32, anchors), get_embeddings(sess_fp32, positives))
cos_int8 = cosine(get_embeddings(sess_int8, anchors), get_embeddings(sess_int8, positives))

drift = np.abs(cos_fp32 - cos_int8).mean()
print(f"Mean cosine drift: {drift:.5f}")   # target < 0.01

Mean cosine drift: 0.07454


In [45]:
# pick 10 natural language queries and see if the right function comes back top-1
queries = [
    "read a file line by line",
    "sort a list of dictionaries by key",
    "connect to a postgresql database",
    "calculate cosine similarity between two vectors",
    "parse command line arguments",
]

corpus_texts  = [eval_dataset[i]["positive"] for i in range(100)]
corpus_embeds = get_embeddings(sess_int8, corpus_texts)

for q in queries:
    q_emb = get_embeddings(sess_int8, [q])
    scores = cosine(
        np.repeat(q_emb, len(corpus_embeds), axis=0),
        corpus_embeds
    )
    top = corpus_texts[scores.argmax()][:120]
    print(f"Q: {q}\nA: {top}\n")

Q: read a file line by line
A: def share_file(comm, path):
    """
    Copies the file from rank 0 to all other ranks
    Puts it in the same place on 

Q: sort a list of dictionaries by key
A: def sample(self, batch_size):
        """Returns a dict {key: array(batch_size x shapes[key])}
        """
        buffe

Q: connect to a postgresql database
A: def logs(self, prefix='worker'):
        """Generates a dictionary that contains all collected statistics.
        """
 

Q: calculate cosine similarity between two vectors
A: def _check_shape(placeholder_shape, data_shape):
    ''' check if two shapes are compatible (i.e. differ only by dimensi

Q: parse command line arguments
A: def parse_cmdline_kwargs(args):
    '''
    convert a list of '='-spaced command-line arguments to a dictionary, evaluat



In [3]:
# Step 1: Login to Hugging Face (run this first)
from huggingface_hub import login

login()  # This will prompt you to enter your HF token

In [ ]:
upload_folder(
    folder_path=".",
    repo_id="KingLLM/nomic-codesearch-onnx",
    repo_type="model",
    ignore_patterns=["blog.md","prepare_android.py", "android_assets/", ".DS_Store","android_src",],
)


Will create 0 deletion commit(s) and 1 addition commit(s), totalling 9 atomic operations.
INFO:huggingface_hub.hf_api.create_commits_on_pr:Will create 0 deletion commit(s) and 1 addition commit(s), totalling 9 atomic operations.
Multi-commits strategy with ID 36f1938d6bd59be836ed9a86e1748d3a8864dac217ea603ed1f58b2c302bc7cd.
INFO:huggingface_hub.hf_api.create_commits_on_pr:Multi-commits strategy with ID 36f1938d6bd59be836ed9a86e1748d3a8864dac217ea603ed1f58b2c302bc7cd.
New PR created: https://huggingface.co/KingLLM/nomic-codesearch-onnx/discussions/1
INFO:huggingface_hub.hf_api.create_commits_on_pr:New PR created: https://huggingface.co/KingLLM/nomic-codesearch-onnx/discussions/1
model.onnx:   0%|          | 0.00/548M [00:00<?, ?B/s]


model.onnx:   0%|          | 16.4k/548M [00:00<1:23:17, 110kB/s]

model.onnx:   0%|          | 197k/548M [00:00<09:47, 932kB/s]   

model.onnx:   0%|          | 508k/548M [00:00<05:31, 1.65MB/s]

model.onnx:   0%|          | 1.10M/548M [00:00<02:56, 3.10MB

HfHubHTTPError: (Request ID: Root=1-6a269e47-33e870517d2f3c4d084dec8f)

403 Forbidden: Authorization error..
Cannot access content at: https://huggingface.co/api/models/KingLLM/nomic-codesearch-onnx/discussions/1/merge.
Make sure your token has the correct permissions.